# Model 1: Random Forest

In [30]:
import pandas as pd
douala = pd.read_csv("../Data/Processed/Douala_Features.csv")

In [31]:
features = [
    "ALLSKY_SFC_SW_DNI",
    "ALLSKY_SFC_SW_DIFF",
    "T2M",
    "RH2M",
    "WS10M",
    "PRECTOTCORR",
    "MO",
    "DY",
    "HR"
]

target = "ALLSKY_SFC_SW_DWN"

In [32]:
from sklearn.model_selection import train_test_split

X = douala[features]
y = douala[target]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

In [33]:
from sklearn.ensemble import RandomForestRegressor

rf = RandomForestRegressor(
    n_estimators=100,
    random_state=42
)

rf.fit(X_train, y_train)

,n_estimators,100
,criterion,'squared_error'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,1.0
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [34]:
y_pred_rf = rf.predict(X_test)

In [35]:
# XGBOOST
from xgboost import XGBRegressor

In [36]:
xgb = XGBRegressor(
    n_estimators=200,
    learning_rate=0.05,
    max_depth=6,
    random_state=42
)

xgb.fit(X_train, y_train)

,objective,'reg:squarederror'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,None
,device,None
,early_stopping_rounds,None
,enable_categorical,False
,eval_metric,None


In [37]:
y_pred_xgb = xgb.predict(X_test)

In [38]:
pip install tensorflow

Note: you may need to restart the kernel to use updated packages.


In [39]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense
from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras.layers import LSTM, Dense, Dropout

In [40]:
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

import numpy as np

In [41]:
def evaluate(y_true, y_pred):

    mae = mean_absolute_error(y_true, y_pred)

    rmse = np.sqrt(
        mean_squared_error(y_true, y_pred)
    )

    r2 = r2_score(y_true, y_pred)

    print("MAE:", mae)
    print("RMSE:", rmse)
    print("R²:", r2)

In [42]:
evaluate(y_test, y_pred_rf)

MAE: 7.368822848721115
RMSE: 15.999058991060982
R²: 0.9955218464311865


In [43]:
evaluate(y_test, y_pred_xgb)

MAE: 6.844247153063828
RMSE: 14.75605224566133
R²: 0.9961906535557523


In [44]:
# Scale data
scaler_X = MinMaxScaler()
scaler_y = MinMaxScaler()

X_scaled = scaler_X.fit_transform(X)
y_scaled = scaler_y.fit_transform(
    y.values.reshape(-1,1)
)

In [45]:
def create_sequences(X, y, seq_length):

    Xs = []
    ys = []

    for i in range(len(X) - seq_length):
        Xs.append(X[i:(i + seq_length)])
        ys.append(y[i + seq_length])

    return np.array(Xs), np.array(ys)

In [46]:
seq_length = 24

In [47]:
X_lstm, y_lstm = create_sequences(
    X_scaled,
    y_scaled,
    seq_length
)

print(X_lstm.shape)
print(y_lstm.shape)

(52560, 24, 9)
(52560, 1)


In [48]:
split_index = int(len(X_lstm) * 0.8)

X_train_lstm = X_lstm[:split_index]
X_test_lstm = X_lstm[split_index:]

y_train_lstm = y_lstm[:split_index]
y_test_lstm = y_lstm[split_index:]

print(X_train_lstm.shape)
print(X_test_lstm.shape)

(42048, 24, 9)
(10512, 24, 9)


In [49]:
model = Sequential()

model.add(
    LSTM(
        64,
        activation="relu",
        input_shape=(
            X_train_lstm.shape[1],
            X_train_lstm.shape[2]
        )
    )
)

model.add(Dropout(0.2))

model.add(Dense(32, activation="relu"))

model.add(Dense(1))

model.compile(
    optimizer="adam",
    loss="mse"
)

model.summary()

c:\Users\user\anaconda3\Lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm_1 (LSTM)                   │ (None, 64)             │        18,944 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 21,057 (82.25 KB)

 Trainable params: 21,057 (82.25 KB)

 Non-trainable params: 0 (0.00 B)

In [50]:
print("mae_rf" in globals())
print("mae_xgb" in globals())
print("mae_lstm" in globals())

True
True
True


In [51]:
mae_rf = mean_absolute_error(y_test, y_pred_rf)

rmse_rf = np.sqrt(
    mean_squared_error(y_test, y_pred_rf)
)

r2_rf = r2_score(y_test, y_pred_rf)

print("Random Forest")
print("MAE =", mae_rf)
print("RMSE =", rmse_rf)
print("R² =", r2_rf)

Random Forest
MAE = 7.368822848721115
RMSE = 15.999058991060982
R² = 0.9955218464311865


In [52]:
mae_xgb = mean_absolute_error(y_test, y_pred_xgb)

rmse_xgb = np.sqrt(
    mean_squared_error(y_test, y_pred_xgb)
)

r2_xgb = r2_score(y_test, y_pred_xgb)

print("XGBoost")
print("MAE =", mae_xgb)
print("RMSE =", rmse_xgb)
print("R² =", r2_xgb)

XGBoost
MAE = 6.844247153063828
RMSE = 14.75605224566133
R² = 0.9961906535557523


In [53]:
y_pred_lstm = model.predict(X_test_lstm)

y_test_actual = scaler_y.inverse_transform(y_test_lstm)

y_pred_actual = scaler_y.inverse_transform(y_pred_lstm)

329/329 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step


In [54]:
mae_lstm = mean_absolute_error(
    y_test_actual,
    y_pred_actual
)

rmse_lstm = np.sqrt(
    mean_squared_error(
        y_test_actual,
        y_pred_actual
    )
)

r2_lstm = r2_score(
    y_test_actual,
    y_pred_actual
)

print("LSTM")
print("MAE =", mae_lstm)
print("RMSE =", rmse_lstm)
print("R² =", r2_lstm)

LSTM
MAE = 179.85644460375866
RMSE = 288.68437260131236
R² = -0.4617952988927805


In [55]:
results = pd.DataFrame({
    "City": "Douala",
    "Model": ["Random Forest", "XGBoost", "LSTM"],
    "MAE": [mae_rf, mae_xgb, mae_lstm],
    "RMSE": [rmse_rf, rmse_xgb, rmse_lstm],
    "R2": [r2_rf, r2_xgb, r2_lstm]
})

results.to_csv("results_douala.csv", index=False)
print(results)

     City          Model         MAE        RMSE        R2
0  Douala  Random Forest    7.368823   15.999059  0.995522
1  Douala        XGBoost    6.844247   14.756052  0.996191
2  Douala           LSTM  179.856445  288.684373 -0.461795


In [56]:
import joblib

joblib.dump(
    xgb,
    "best_douala_xgb.pkl"
)

print("Douala model saved successfully")

Douala model saved successfully


In [57]:
importance = xgb.feature_importances_

feature_importance = pd.DataFrame({
    "Feature": features,
    "Importance": importance
})

feature_importance = feature_importance.sort_values(
    by="Importance",
    ascending=False
)

print(feature_importance)

              Feature  Importance
1  ALLSKY_SFC_SW_DIFF    0.919683
0   ALLSKY_SFC_SW_DNI    0.059848
8                  HR    0.014736
6                  MO    0.002138
2                 T2M    0.001480
3                RH2M    0.000742
5         PRECTOTCORR    0.000591
7                  DY    0.000498
4               WS10M    0.000285
